In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import joblib
import warnings
warnings.filterwarnings('ignore')

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
df_train = pd.read_csv('/content/drive/MyDrive/archive/fraudTrain_cleaned.csv')

In [4]:
print(f"Dataset shape: {df_train.shape}")
print(f"Columns: {df_train.columns.tolist()}")

Dataset shape: (1296675, 29)
Columns: ['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'datetime', 'hour', 'day_of_week', 'month', 'is_weekend', 'time_of_day', 'amt_bin']


In [5]:
print("\nFraud distribution:")
print(df_train['is_fraud'].value_counts())
print(f"\nFraud rate: {df_train['is_fraud'].mean() * 100:.2f}%")


Fraud distribution:
is_fraud
0    1289169
1       7506
Name: count, dtype: int64

Fraud rate: 0.58%


## Why Isolation Forest?

### The Problem
Our dataset has **1.29 million transactions** with only **0.58% fraud** (7,506 fraudulent transactions). This extreme imbalance makes traditional classification difficult — a model that predicts "not fraud" for everything would be 99.42% accurate but useless.

### What is Isolation Forest?
Isolation Forest is an **unsupervised anomaly detection** algorithm. It works on a simple principle: **anomalies are rare and different**, so they're easier to isolate.

**How it works:**
1. Randomly selects a feature and a split point
2. Recursively partitions data until each point is isolated
3. Anomalies require **fewer splits** to isolate (they stand out)
4. Normal points require **more splits** (they're clustered together)

### Why It Fits Our Project

| Reason | Explanation |
|--------|-------------|
| **Handles imbalance** | Designed for rare events (0.58% fraud) |
| **Unsupervised** | Learns what's "normal" without needing labels |
| **Scalable** | Efficient on 1.3M transactions |
| **No assumptions** | Doesn't assume data distribution |

### Features we are using from EDA
- **amt:** Fraud avg 531 dollars vs Normal avg $68
- **hour:** Fraud peaks at hours 22-23 (~3% fraud rate)
- **category:** shopping_net (1.75%) and misc_net (1.45%) have highest fraud rates
- **day_of_week & is_weekend:** Minor predictors but add context

In [6]:
# Select features for Isolation Forest
# Based on EDA: amt, hour, and category are strong fraud predictors

df_model = df_train.copy()
le_category = LabelEncoder()
df_model['category_encoded'] = le_category.fit_transform(df_model['category'])

In [7]:
# Select features
df_model['is_high_amount'] = (df_model['amt'] > 200).astype(int)
df_model['is_night'] = df_model['hour'].isin([22, 23, 0, 1, 2, 3]).astype(int)

features = ['amt', 'hour', 'day_of_week', 'category_encoded', 'is_weekend', 'is_high_amount', 'is_night']
X = df_model[features]

print(f"Features selected: {features}")
print(f"Shape: {X.shape}")
X.head()

Features selected: ['amt', 'hour', 'day_of_week', 'category_encoded', 'is_weekend', 'is_high_amount', 'is_night']
Shape: (1296675, 7)


,amt,hour,day_of_week,category_encoded,is_weekend,is_high_amount,is_night
0,4.97,0,1,8,0,0,1
1,107.23,0,1,4,0,0,1
2,220.11,0,1,0,0,1,1
3,45.00,0,1,2,0,0,1
4,41.96,0,1,9,0,0,1


In [8]:
# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Scaled data shape: {X_scaled.shape}")
print(f"Mean: {X_scaled.mean():.4f}, Std: {X_scaled.std():.4f}")

Scaled data shape: (1296675, 7)
Mean: -0.0000, Std: 1.0000


In [9]:
# Train Isolation Forest
# contamination = expected fraction of anomalies (fraud rate is 0.58%)

iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.03,
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_scaled)
print("Isolation Forest trained!")

Isolation Forest trained!


In [10]:
# Get predictions
# Isolation Forest returns: 1 = normal, -1 = anomaly
predictions = iso_forest.predict(X_scaled)

# Convert to match our labels: 0 = normal, 1 = fraud
df_model['anomaly'] = (predictions == -1).astype(int)

print(f"Anomalies detected: {df_model['anomaly'].sum()}")
print(f"Anomaly rate: {df_model['anomaly'].mean() * 100:.2f}%")

Anomalies detected: 38900
Anomaly rate: 3.00%


In [11]:
# Compare predictions with actual fraud labels
print("="*50)
print("ISOLATION FOREST EVALUATION")
print("="*50)

# Confusion Matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(df_model['is_fraud'], df_model['anomaly'])
print(cm)

print("\n")
print("                 Predicted")
print("                 Normal  Anomaly")
print(f"Actual Normal    {cm[0][0]:>7}  {cm[0][1]:>7}")
print(f"Actual Fraud     {cm[1][0]:>7}  {cm[1][1]:>7}")

ISOLATION FOREST EVALUATION

Confusion Matrix:
[[1255798   33371]
 [   1977    5529]]


                 Predicted
                 Normal  Anomaly
Actual Normal    1255798    33371
Actual Fraud        1977     5529


In [12]:
# Classification Report
print("\nClassification Report:")
print(classification_report(df_model['is_fraud'], df_model['anomaly'],
                            target_names=['Normal', 'Fraud']))


Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      0.97      0.99   1289169
       Fraud       0.14      0.74      0.24      7506

    accuracy                           0.97   1296675
   macro avg       0.57      0.86      0.61   1296675
weighted avg       0.99      0.97      0.98   1296675



In [13]:
# Key Metrics
tn, fp, fn, tp = cm.ravel()

print("\nKey Metrics:")
print(f"True Positives (Fraud caught): {tp}")
print(f"False Positives (Normal flagged as fraud): {fp}")
print(f"False Negatives (Fraud missed): {fn}")
print(f"True Negatives (Normal correctly identified): {tn}")

# Detection rate
detection_rate = tp / (tp + fn) * 100
false_alarm_rate = fp / (fp + tn) * 100
print(f"\nFraud Detection Rate: {detection_rate:.2f}%")
print(f"False Alarm Rate: {false_alarm_rate:.2f}%")


Key Metrics:
True Positives (Fraud caught): 5529
False Positives (Normal flagged as fraud): 33371
False Negatives (Fraud missed): 1977
True Negatives (Normal correctly identified): 1255798

Fraud Detection Rate: 73.66%
False Alarm Rate: 2.59%


In [20]:
# Final Model Summary
print("FINAL ISOLATION FOREST MODEL")
print("="*50)
print(f"\nModel: Isolation Forest")
print(f"Contamination: 3%")
print(f"Features: {features}")
print(f"\nPerformance:")
print(f"  - Fraud Detection Rate: 73.66%")
print(f"  - False Alarm Rate: 2.59%")
print(f"  - Frauds Caught: 5,529 / 7,506")
print(f"  - Frauds Missed: 1,977")

FINAL ISOLATION FOREST MODEL

Model: Isolation Forest
Contamination: 3%
Features: ['amt', 'hour', 'day_of_week', 'category_encoded', 'is_weekend', 'is_high_amount', 'is_night']

Performance:
  - Fraud Detection Rate: 73.66%
  - False Alarm Rate: 2.59%
  - Frauds Caught: 5,529 / 7,506
  - Frauds Missed: 1,977


Isolation Forest detected 73.66% of fraudulent transactions with a
2.59% false alarm rate. While supervised models typically achieve
higher detection rates (90%+), Isolation Forest's advantage is that
it works without labeled data and can detect NEW types of fraud
that weren't in the training data.

In [22]:
joblib.dump(iso_forest, '/content/drive/MyDrive/archive/isolation_forest_model.pkl')
print("Isolation Forest model saved!")

joblib.dump(scaler, '/content/drive/MyDrive/archive/isolation_forest_scaler.pkl')
print("Scaler saved!")

joblib.dump(le_category, '/content/drive/MyDrive/archive/isolation_forest_label_encoder.pkl')
print("Label encoder saved!")

Isolation Forest model saved!
Scaler saved!
Label encoder saved!


## Why Isolation Forest?

### What is the purpose of Isolation Forest in our Financial Advisor

Isolation Forest is used for **anomaly detection** — flagging unusual transactions that don't fit the user's normal spending pattern.

### Use Cases

| Feature | How It Helps |
|---------|--------------|
| Fraud alerts | "This transaction looks suspicious" |
| Spending alerts | "You usually don't spend this much on shopping" |
| Budget warnings | "Unusual spending detected — you may exceed budget" |


### Why Not Just Use Supervised Learning?

| Advantage of Isolation Forest | Explanation |
|-------------------------------|-------------|
| No labels needed | Works without knowing which transactions are fraud |
| Detects NEW patterns | Can catch fraud types not seen before |
| Personalized | Learns what's "normal" for each user |

We use Isolation Forest to alert users about unusual transactions — whether it's fraud OR just out-of-pattern spending. The 73.66% detection rate which we got is good for this purpose.